# Day 02 · Lines — The Half-Mile Promise

**Question.** Chicago's grid puts an arterial street every half mile, and a bus on nearly every one.
Which bus routes carry the city's riders, and how often does the bus come?

**Approach.** Average weekday riders per route over the last twelve months of CTA's open data, joined
to the scheduled service on one reference weekday from CTA's GTFS feed: route shapes, trips, vehicle-hours
and the midday interval. Shapes are projected to Chicago address units, cut into short pieces, and pieces
that share a street are grouped so the site can draw routes on a shared street as offset parallels.

**Output.** `output/plate.json`, read by the website to draw the plate.

| Section | What happens |
| --- | --- |
| 1. Setup | Imports, paths, run settings |
| 2. Pull | Dated snapshot of route ridership (cached); the GTFS feed already in `reference/raw/gtfs/` |
| 3. Validate | Schema, values and consistency checks that stop the run on surprises |
| 4. Transform | Weekday riders per route; service per route; one shape per route; shared streets |
| 5. Analyze | Concentration, frequency, productivity, quarter-mile coverage |
| 6. QA view | Quick charts to catch errors before publishing |
| 7. Export | `plate.json` with the data ledger |

Run top to bottom. To refresh the ridership data, set `REFRESH = True` in section 1. To refresh the
GTFS feed, run `uv run python -m atlas.gtfs --refresh` from the repository root.

## 1. Setup

In [ ]:
from __future__ import annotations

import json
import logging
import zipfile
from collections import defaultdict
from datetime import date
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.path import Path as MplPath

from atlas import Ledger, snapshot, write_plate
from atlas.gtfs import bus_trips, shape_paths

logging.basicConfig(level=logging.INFO, format='%(levelname)s %(name)s: %(message)s')
log = logging.getLogger('day02')
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 140)

In [ ]:
# Run settings ---------------------------------------------------------------------------------
REFRESH = False  # True = pull a new ridership snapshot from the Chicago Data Portal

DAY = 2
DATASET_ID = 'jyb9-n7fm'  # CTA - Ridership - Bus Routes - Daily Totals by Route
DATASET_NAME = 'CTA - Ridership - Bus Routes - Daily Totals by Route'
DATASET_URL = f'https://data.cityofchicago.org/d/{DATASET_ID}'
GTFS_URL = 'https://www.transitchicago.com/developers/gtfs/'

# Paths: the notebook runs from its own folder (days/day02-lines/)
DAY_DIR = Path.cwd()
REPO = DAY_DIR.parents[1]
RAW_DIR = DAY_DIR / 'data' / 'raw'
OUT_PATH = DAY_DIR / 'output' / 'plate.json'
GTFS_ZIP = sorted((REPO / 'reference' / 'raw' / 'gtfs').glob('cta_gtfs_*.zip'))[-1]
BOUNDARY = json.loads((REPO / 'reference' / 'city-boundary.json').read_text())
assert (DAY_DIR / 'day02.ipynb').exists(), f'Run this notebook from its own folder, not {DAY_DIR}'

# Analysis constants ---------------------------------------------------------------------------
BENCHMARK_YEAR = 2019            # the last full year before the pandemic
REF_DAY = date(2026, 10, 14)     # a Wednesday inside the feed's calendar, no holiday exceptions
MIDDAY = (10 * 3600, 15 * 3600)  # midday window for the interval between buses
TIERS = [                        # midday interval, minutes (upper bound inclusive)
    ('frequent', 'every 10 min or better', 10),
    ('regular', 'every 11–15 min', 15),
    ('basic', 'less often than every 15 min', np.inf),
]
NO_MIDDAY = ('none', 'no midday service')
QUARTER_MILE = 200               # address units (800 per mile)

# Geometry constants (address units)
SIMPLIFY = 10                    # Douglas-Peucker tolerance for shapes, about 66 ft
STEP = 40                        # piece length for finding shared streets, 1/20 mile

## 2. Pull

The SODA API returns **one row per route per day**, with the day type (`W` weekday, `A` Saturday,
`U` Sunday or holiday). Only the benchmark year and the last two calendar years are pulled; the
twelve-month window is cut from them in section 4. The GTFS feed is the one already used for
Day 01's station order (`reference/raw/gtfs/`, kept out of git).

In [ ]:
raw, pulled = snapshot(
    DATASET_ID,
    RAW_DIR,
    'bus_route_daily',
    refresh=REFRESH,
    select='route,date,daytype,rides',
    where=f"date between '{BENCHMARK_YEAR}-01-01' and '{BENCHMARK_YEAR}-12-31' or date >= '2025-01-01'",
    order='date,route',
)
log.info('Snapshot pulled %s: %d rows x %d columns', pulled, *raw.shape)

feed_date = max(date(*i.date_time[:3]) for i in zipfile.ZipFile(GTFS_ZIP).infolist()
                if i.filename == 'stop_times.txt')
trips = bus_trips(GTFS_ZIP, REF_DAY)
routes_gtfs = pd.read_csv(zipfile.ZipFile(GTFS_ZIP).open('routes.txt'), dtype=str)
routes_gtfs = routes_gtfs[routes_gtfs['route_type'] == '3'].set_index('route_id')
log.info('GTFS %s (feed of %s): %d bus trips on %s across %d routes',
         GTFS_ZIP.name, feed_date, len(trips), REF_DAY, trips['route_id'].nunique())
raw.head()

## 3. Validate

In [ ]:
EXPECTED = {'route', 'date', 'daytype', 'rides'}
missing = EXPECTED - set(raw.columns)
assert not missing, f'Missing columns: {missing}'

rides = raw[list(EXPECTED)].copy()
rides['date'] = pd.to_datetime(rides['date'])
rides['rides'] = pd.to_numeric(rides['rides'], errors='coerce')
rides['route'] = rides['route'].str.strip()

checks = {
    'no missing route': rides['route'].notna().all(),
    'rides parse': rides['rides'].notna().all(),
    'rides are non-negative': (rides['rides'] >= 0).all(),
    'day type is W/A/U': rides['daytype'].isin({'W', 'A', 'U'}).all(),
    'one row per route and day': not rides.duplicated(['route', 'date']).any(),
    'one day type per date': (rides.groupby('date')['daytype'].nunique() == 1).all(),
    'weekdays are Mon-Fri': (rides.loc[rides['daytype'] == 'W', 'date'].dt.day_of_week < 5).all(),
    'reference day is a Wednesday': REF_DAY.weekday() == 2,
    'reference day has bus trips': len(trips) > 10_000,
}
for name, ok in checks.items():
    log.info('%-32s %s', name, 'ok' if ok else 'FAILED')
assert all(checks.values()), 'Validation failed; see log above'

In [ ]:
# The twelve-month window ends on the last day in the data, which should close a month; a partial
# month at the end would bias the average toward one season.
last = rides['date'].max()
assert last == last + pd.offsets.MonthEnd(0), f'Data ends mid-month ({last.date()}); trim the window'
WINDOW = (last - pd.DateOffset(years=1) + pd.Timedelta(days=1), last)
window_days = pd.date_range(*WINDOW)
have = set(rides.loc[rides['date'].between(*WINDOW), 'date'])
gaps = [d.date() for d in window_days if d not in have]
assert not gaps, f'Days with no ridership in the window: {gaps[:10]}'
log.info('Window %s to %s: %d days, all present', WINDOW[0].date(), WINDOW[1].date(), len(window_days))

## 4. Transform

- **Average weekday riders.** All weekday rides on a route in the window, divided by the number of
  weekdays in the window. A route that runs only part of the year (a seasonal or event route) is
  averaged over the whole year, so its share of the year's riders is not overstated.
- **Service.** Every bus trip scheduled on the reference Wednesday: trips, vehicle-hours (first to
  last stop, layover excluded) and the midday interval: 300 minutes divided by the average number of
  trips per direction starting between 10:00 and 15:00.
- **Shape.** One shape per route: in each direction, the longest shape that carries at least a
  quarter of that direction's trips (so short-turns and garage trips are skipped); then the longer of
  the two directions. Where the other direction runs on a different street (a one-way pair, a loop at
  the end), those stretches are kept too.
- **'L' lines.** Rail shapes from the same feed, drawn on the map for context only.
- **Shared streets.** Each shape is simplified and cut into 1/20-mile pieces. Pieces of different
  routes that run parallel within about 130 ft share a street; the routes side by side at a piece
  form its *bundle*. Bundles lasting under a quarter mile along a route are smoothed away. Bundles
  feed the busiest-stretch table; the site draws shared streets once, most frequent route on top.

In [ ]:
def weekday_average(df: pd.DataFrame, start: pd.Timestamp, end: pd.Timestamp) -> pd.Series:
    """Average weekday riders per route: weekday rides over the period / weekdays in the period."""
    wk = df[df['date'].between(start, end) & (df['daytype'] == 'W')]
    return wk.groupby('route')['rides'].sum() / wk['date'].nunique()


riders = weekday_average(rides, *WINDOW).rename('riders')
riders_bench = weekday_average(rides, pd.Timestamp(f'{BENCHMARK_YEAR}-01-01'),
                               pd.Timestamp(f'{BENCHMARK_YEAR}-12-31')).rename('riders_bench')
log.info('Weekday riders, window: %s on %d routes | %d: %s on %d routes',
         f'{riders.sum():,.0f}', len(riders), BENCHMARK_YEAR, f'{riders_bench.sum():,.0f}', len(riders_bench))

In [ ]:
# Routes in the ridership data but not scheduled on the reference day (event, seasonal and shuttle
# routes) stay in the totals but cannot be drawn. Every scheduled route must have ridership.
scheduled = set(trips['route_id'])
not_scheduled = riders[~riders.index.isin(scheduled)].sort_values(ascending=False)
no_riders = sorted(scheduled - set(riders.index))
assert not no_riders, f'Scheduled routes with no ridership: {no_riders}'
log.info('Not scheduled on %s: %s (%.2f%% of weekday riders)', REF_DAY,
         not_scheduled.round().astype(int).to_dict(), 100 * not_scheduled.sum() / riders.sum())

In [ ]:
def interval(g: pd.DataFrame) -> float:
    """Midday minutes between buses: 300 / average trips per direction starting 10:00-15:00."""
    mid = g[(g['start'] >= MIDDAY[0]) & (g['start'] < MIDDAY[1])]
    if mid.empty:
        return np.nan
    per_dir = mid.groupby('direction_id').size().mean()
    return (MIDDAY[1] - MIDDAY[0]) / 60 / per_dir


def tier(minutes: float) -> str:
    if np.isnan(minutes):
        return 'none'  # no midday trips: rush-hour, night or event service only
    return next(key for key, _, upper in TIERS if minutes <= upper)


service = trips.assign(hours=(trips['end'] - trips['start']) / 3600).groupby('route_id').apply(
    lambda g: pd.Series({
        'trips': len(g),
        'veh_hours': g['hours'].sum(),
        'interval': interval(g),
        'midday_bph': ((g['start'] >= MIDDAY[0]) & (g['start'] < MIDDAY[1])).sum() / 5,
        'first': g['start'].min() / 3600,
        'last': g['end'].max() / 3600,
    }),
    include_groups=False,
)
service['tier'] = service['interval'].map(tier)
service['name'] = routes_gtfs['route_long_name'].reindex(service.index)
service.sort_values('interval').head()

In [ ]:
def direction_shapes(g: pd.DataFrame) -> list[str]:
    """Per direction, the longest shape carrying >= 25% of that direction's trips."""
    best = []
    for _, d in g.groupby('direction_id'):
        counts = d['shape_id'].value_counts()
        common = counts[counts >= 0.25 * counts.sum()].index
        best.append(max(common, key=lambda s: shape_len[s]))
    return best


def representative_shape(g: pd.DataFrame) -> str:
    """The longer of the two directions' shapes: the line drawn for the route."""
    return max(direction_shapes(g), key=lambda s: shape_len[s])


paths = shape_paths(GTFS_ZIP, set(trips['shape_id']))
shape_len = {sid: p['dist'].iloc[-1] for sid, p in paths.items()}  # feet along the shape
rep = trips.groupby('route_id').apply(representative_shape, include_groups=False).rename('shape_id')
dir_shapes = trips.groupby('route_id').apply(direction_shapes, include_groups=False)
log.info('Representative shapes: %d routes', len(rep))

In [ ]:
def simplify(pts: np.ndarray, tol: float) -> np.ndarray:
    """Douglas-Peucker: keep the points needed to stay within ``tol`` of the original line."""
    keep = np.zeros(len(pts), bool)
    keep[[0, -1]] = True
    stack = [(0, len(pts) - 1)]
    while stack:
        a, b = stack.pop()
        if b <= a + 1:
            continue
        seg = pts[b] - pts[a]
        norm = np.hypot(*seg) or 1.0
        rel = pts[a + 1:b] - pts[a]
        d = np.abs(seg[0] * rel[:, 1] - seg[1] * rel[:, 0]) / norm
        i = int(np.argmax(d))
        if d[i] > tol:
            keep[a + 1 + i] = True
            stack += [(a, a + 1 + i), (a + 1 + i, b)]
    return pts[keep]


def densify(pts: np.ndarray, step: float) -> np.ndarray:
    """Points every ``step`` units along the line (plus its end)."""
    cum = np.r_[0, np.cumsum(np.hypot(*np.diff(pts, axis=0).T))]
    s = np.unique(np.r_[np.arange(0, cum[-1], step), cum[-1]])
    return np.c_[np.interp(s, cum, pts[:, 0]), np.interp(s, cum, pts[:, 1])]


lines = {r: densify(simplify(paths[s][['x', 'y']].to_numpy(), SIMPLIFY), STEP) for r, s in rep.items()}
length_mi = pd.Series({r: np.hypot(*np.diff(p, axis=0).T).sum() / 800 for r, p in lines.items()})
log.info('Route length: %.0f mi drawn, median %.1f mi', length_mi.sum(), length_mi.median())

In [ ]:
# The other direction, where it runs on a different street: one-way pairs, loops at the ends. Its
# shape is kept only where it lies more than APART units (about 200 ft) from the drawn line for at
# least MIN_APART pieces, so the map draws both streets of a pair and the coverage test counts both.
APART, MIN_APART = 30, 4


def apart_runs(p: np.ndarray, base: np.ndarray) -> list[np.ndarray]:
    """Stretches of ``p`` farther than APART from ``base``, each with one point either side."""
    far = np.sqrt(((p[:, None, :] - base[None, :, :]) ** 2).sum(-1)).min(axis=1) > APART
    edges = np.flatnonzero(np.diff(np.r_[0, far.astype(int), 0]))
    return [p[max(a - 1, 0):min(b + 1, len(p))] for a, b in zip(edges[::2], edges[1::2]) if b - a >= MIN_APART]


alt = {}
for r, shapes in dir_shapes.items():
    runs = [run for s in shapes if s != rep[r]
            for run in apart_runs(densify(simplify(paths[s][['x', 'y']].to_numpy(), SIMPLIFY), STEP), lines[r])]
    if runs:
        alt[r] = runs
alt_mi = sum(np.hypot(*np.diff(run, axis=0).T).sum() for runs in alt.values() for run in runs) / 800
log.info('Other direction on another street: %d routes, %.0f mi', len(alt), alt_mi)

In [ ]:
# 'L' lines, for context on the map: every rail shape that carries at least a tenth of its line's
# trips (so both Green Line branches and the Purple express to the Loop are included).
z = zipfile.ZipFile(GTFS_ZIP)
rail_routes = pd.read_csv(z.open('routes.txt'), dtype=str).query('route_type == "1"')
rail_trips = pd.read_csv(z.open('trips.txt'), dtype=str, usecols=['route_id', 'shape_id'])
rail_trips = rail_trips[rail_trips['route_id'].isin(rail_routes['route_id'])]
counts = rail_trips.groupby(['route_id', 'shape_id']).size()
counts = counts[counts >= 0.1 * counts.groupby(level=0).transform('sum')]
rail_paths = shape_paths(GTFS_ZIP, set(counts.index.get_level_values(1)))
rail = [{'line': rid, 'path': [[round(x), round(y)] for x, y in simplify(rail_paths[sid][['x', 'y']].to_numpy(), SIMPLIFY)]}
        for rid, sid in counts.index]
log.info('Rail shapes: %d on %d lines, %d vertices', len(rail), counts.index.get_level_values(0).nunique(),
         sum(len(r['path']) for r in rail))

In [ ]:
# Shared streets. A piece of one route runs beside a piece of another when they are parallel (within
# 12 degrees), overlap along the street, and lie within LATERAL units of each other across it, about
# 130 ft: close enough for one street's shapes, far enough apart to keep State, Wabash and Michigan
# separate. Each piece's bundle is its route plus the routes beside it. Along each route, bundles
# that last less than MIN_RUN pieces (a crossing, a turn, a block shared in passing) take the bundle
# before them, so the drawn lines do not jitter.
LATERAL, PARALLEL, MIN_RUN = 20, np.radians(12), 5
pieces = []
for r, p in lines.items():
    d = p[1:] - p[:-1]
    u = d / np.hypot(*d.T)[:, None]
    flip = (u[:, 0] < -1e-9) | ((np.abs(u[:, 0]) <= 1e-9) & (u[:, 1] < 0))
    u[flip] *= -1  # canonical direction: eastward, or northward when vertical
    mid = (p[:-1] + p[1:]) / 2
    pieces.append(pd.DataFrame({'route': r, 'i': np.arange(len(mid)), 'mx': mid[:, 0], 'my': mid[:, 1],
                                'ux': u[:, 0], 'uy': u[:, 1]}))
pieces = pd.concat(pieces, ignore_index=True)
P = pieces[['mx', 'my']].to_numpy()
U = pieces[['ux', 'uy']].to_numpy()
R = pieces['route'].to_numpy()

HASH = 40
buckets = defaultdict(list)
for k, (x, y) in enumerate(np.floor(P / HASH).astype(int)):
    buckets[(x, y)].append(k)
beside = [{R[k]} for k in range(len(pieces))]
for (cx, cy), ks in buckets.items():
    near = np.array([j for dx in (-1, 0, 1) for dy in (-1, 0, 1) for j in buckets.get((cx + dx, cy + dy), ())])
    for k in ks:
        cand = near[R[near] != R[k]]
        if not len(cand):
            continue
        rel = P[cand] - P[k]
        along = np.abs(rel @ U[k])
        across = np.abs(rel[:, 0] * U[k][1] - rel[:, 1] * U[k][0])
        parallel = np.abs(U[cand] @ U[k]) > np.cos(PARALLEL)
        beside[k].update(R[cand[(along <= STEP / 2) & (across <= LATERAL) & parallel]])
pieces['local'] = [tuple(sorted(s)) for s in beside]


def smooth(sets: list[tuple[str, ...]]) -> list[tuple[str, ...]]:
    """Replace runs shorter than MIN_RUN with the run before (or after, at the start)."""
    sets = list(sets)
    while True:
        runs, start = [], 0
        for i in range(1, len(sets) + 1):
            if i == len(sets) or sets[i] != sets[start]:
                runs.append((start, i))
                start = i
        short = [(a, b) for a, b in runs if b - a < MIN_RUN]
        if not short or len(runs) == 1:
            return sets
        a, b = min(short, key=lambda ab: ab[1] - ab[0])
        fill = sets[a - 1] if a > 0 else sets[b]
        sets[a:b] = [fill] * (b - a)


pieces = pieces.sort_values(['route', 'i'], ignore_index=True)
pieces['set'] = pieces.groupby('route', sort=False)['local'].transform(lambda s: pd.Series(smooth(s.tolist()), index=s.index))
shared = pieces[pieces['set'].str.len() > 1]
log.info('Pieces %d; %.0f%% run beside another route; %d distinct bundles; most routes side by side: %d',
         len(pieces), 100 * len(shared) / len(pieces), shared['set'].nunique(), pieces['set'].str.len().max())
pieces.loc[pieces['set'].str.len() >= 8].groupby('set').size().sort_values(ascending=False).head(8)

## 5. Analyze

In [ ]:
table = service.join(riders).join(riders_bench).join(length_mi.rename('length_mi'))
table['per_hour'] = table['riders'] / table['veh_hours']
table['recovery'] = table['riders'] / table['riders_bench']
table = table.sort_values('riders', ascending=False)
table['rank'] = np.arange(1, len(table) + 1)
table['cum_share'] = table['riders'].cumsum() / table['riders'].sum()

total_riders = riders.sum()  # every route in the data, drawn or not
n_half = int((table['riders'].cumsum() < total_riders / 2).sum() + 1)
by_tier = table.groupby('tier').agg(routes=('riders', 'size'), riders=('riders', 'sum'), length=('length_mi', 'sum'))
by_tier['share'] = by_tier['riders'] / total_riders

summary = {
    'routes': len(table),
    'routesInData': len(riders),
    'riders': round(total_riders),
    'ridersBench': round(riders_bench.sum()),
    'recovery': round(float(total_riders / riders_bench.sum()), 3),
    'halfRoutes': n_half,
    'top10Share': round(float(table['riders'].head(10).sum() / total_riders), 3),
    'trips': len(trips),
    'vehHours': round(table['veh_hours'].sum()),
    'perHour': round(float(total_riders / table['veh_hours'].sum()), 1),
    'tiers': {t: {'routes': int(by_tier.loc[t, 'routes']), 'share': round(float(by_tier.loc[t, 'share']), 3)}
              for t in by_tier.index},
}
log.info('Summary: %s', {k: v for k, v in summary.items() if k != 'tiers'})
by_tier

In [ ]:
table[['name', 'riders', 'riders_bench', 'recovery', 'interval', 'tier', 'per_hour', 'length_mi', 'cum_share']].head(30).round(2)

In [ ]:
# Productivity: the routes that carry the most riders per hour of bus time, and the least. Riders are
# the window's weekday average; hours are the reference Wednesday's schedule.
cols = ['name', 'riders', 'veh_hours', 'per_hour', 'interval', 'tier']
pd.concat([table.nlargest(8, 'per_hour')[cols], table.nsmallest(8, 'per_hour')[cols]]).round(1)

In [ ]:
# The half-mile promise, tested as straight-line distance: the share of the city within a quarter mile
# of a bus route (the farthest a rider is from the nearest of two arterials half a mile apart), and
# within a quarter mile of a route that comes every 10 minutes or better at midday. Both directions
# count where they run on different streets.
RES = 25
rings = [np.array(r, float) for r in BOUNDARY['rings']]
allpts = np.vstack(rings)
x0, y0 = allpts.min(axis=0)
x1, y1 = allpts.max(axis=0)
gx = np.arange(x0, x1, RES) + RES / 2
gy = np.arange(y0, y1, RES) + RES / 2
cells = np.c_[np.repeat(gx, len(gy)), np.tile(gy, len(gx))]
inside = np.zeros(len(cells), bool)
for ring in rings:  # even-odd rule, as the plates draw it
    inside ^= MplPath(ring).contains_points(cells)
inside = inside.reshape(len(gx), len(gy))

r_cells = int(QUARTER_MILE / RES)
offsets = [(i, j) for i in range(-r_cells, r_cells + 1) for j in range(-r_cells, r_cells + 1)
           if (i * i + j * j) * RES * RES <= QUARTER_MILE ** 2]


def near(route_ids) -> np.ndarray:
    """Cells within a quarter mile (straight line) of any of the routes."""
    hit = np.zeros((len(gx) + 2 * r_cells, len(gy) + 2 * r_cells), bool)
    for p in (densify(q, RES / 2) for r in route_ids for q in [lines[r], *alt.get(r, [])]):
        i = np.clip(((p[:, 0] - x0) // RES).astype(int), 0, len(gx) - 1) + r_cells
        j = np.clip(((p[:, 1] - y0) // RES).astype(int), 0, len(gy) - 1) + r_cells
        hit[i, j] = True
    out = np.zeros((len(gx), len(gy)), bool)
    for di, dj in offsets:
        out |= hit[r_cells + di:r_cells + di + len(gx), r_cells + dj:r_cells + dj + len(gy)]
    return out


near_any = near(table.index)
near_freq = near(table.index[table['tier'] == 'frequent'])
coverage = {
    'cityMi2': round(float(inside.sum() * RES * RES / 800 ** 2), 1),
    'anyShare': round(float((near_any & inside).sum() / inside.sum()), 3),
    'frequentShare': round(float((near_freq & inside).sum() / inside.sum()), 3),
}
log.info('Coverage: %s', coverage)

In [ ]:
# Busiest stretches at midday: buses an hour on the routes running side by side at each piece,
# summed over those routes (each route is drawn one way; the count covers both directions).
bph = table['midday_bph']
stretch = pieces.groupby('local').agg(n=('i', 'size'), x=('mx', 'mean'), y=('my', 'mean'))
stretch['routes'] = stretch.index.str.len()
stretch = stretch[stretch['routes'] > 1]
stretch['length_mi'] = stretch['n'] * STEP / 800 / stretch['routes']
stretch['bph'] = [sum(bph.get(r, 0) for r in s) for s in stretch.index]
busiest = stretch[stretch['length_mi'] >= 0.5].nlargest(10, 'bph')
busiest[['routes', 'bph', 'length_mi', 'x', 'y']].round(1)

In [ ]:
# The finding printed on the plate, generated from the data so it can never drift from the numbers.
freq = summary['tiers']['frequent']
FINDING = (
    f'{n_half} of {len(riders)} bus routes carry half of Chicago’s weekday bus riders. '
    f'{freq["routes"]} routes come every 10 minutes or better at midday, and they carry '
    f'{freq["share"]:.0%} of riders.'
)
print(FINDING)

## 6. QA view

A plain check, not the final design: routes in address space (width = weekday riders, value = midday
tier), the cumulative share of riders by route rank, and quarter-mile coverage. Look for routes far off
the network, a route drawn as a short-turn, or an implausible curve.

In [ ]:
shade = {'frequent': '0.05', 'regular': '0.4', 'basic': '0.65', 'none': '0.8'}
fig, (ax_map, ax_cum, ax_cov) = plt.subplots(1, 3, figsize=(16, 8), gridspec_kw={'width_ratios': [1, 1, 1]})
for r, row in table.iloc[::-1].iterrows():
    p = lines[r]
    ax_map.plot(p[:, 0], p[:, 1], color=shade[row['tier']], lw=0.3 + 3.5 * row['riders'] / table['riders'].max(),
                solid_capstyle='butt')
for r in table.index[:15]:
    p = lines[r]
    ax_map.annotate(r, p[len(p) // 2], fontsize=7, color='crimson')
ax_map.set_aspect('equal')
ax_map.set(title='Routes: width = weekday riders, value = midday tier', xlabel='west (-) / east (+)')

ax_cum.plot(table['rank'], table['cum_share'], color='0.15')
ax_cum.axhline(0.5, color='0.7', lw=0.6)
ax_cum.axvline(n_half, color='0.7', lw=0.6)
ax_cum.set(title=f'Half the riders on {n_half} routes', xlabel='routes, busiest first', ylabel='cumulative share')

ax_cov.imshow((inside * 1 + (near_any & inside) + (near_freq & inside)).T, origin='lower', cmap='Greys',
              extent=(x0, x1, y0, y1))
ax_cov.set(title='Within 1/4 mi: any route (mid), frequent (dark)')
for ax in (ax_cum,):
    for side in ('top', 'right'):
        ax.spines[side].set_visible(False)
fig.tight_layout()
plt.show()

In [ ]:
# Bundles in the Loop, one colour per bundle: a street should hold one colour between turns and joins.
fig, ax = plt.subplots(figsize=(8, 8))
loop = pieces[pieces['mx'].between(-1200, 800) & pieces['my'].between(-1200, 1200)]
for g, d in loop.groupby('set'):
    ax.scatter(d['mx'], d['my'], s=3, color=plt.cm.tab20(hash(g) % 20))
ax.set_aspect('equal')
ax.set(title='Street groups, Loop')
plt.show()

## 7. Export

Keys are camelCase because the website reads them directly. Each route's path is a list of
`[x, y, bundle]` in address units: `bundle` indexes the routes sharing the street from that vertex
to the next (`-1` when the route runs alone). Vertices are kept where the bundle changes or the line
bends, so straight runs on one street collapse to their ends.

In [ ]:
bundles: list[tuple[str, ...]] = []
bundle_index: dict[tuple[str, ...], int] = {}


def bundle_of(route_set: tuple[str, ...]) -> int:
    if len(route_set) < 2:
        return -1
    if route_set not in bundle_index:
        bundle_index[route_set] = len(bundles)
        bundles.append(route_set)
    return bundle_index[route_set]


def export_path(r: str) -> list[list[int]]:
    p = lines[r]
    sets = pieces.loc[pieces['route'] == r].sort_values('i')['set'].map(bundle_of).to_numpy()
    d = np.diff(p, axis=0)
    ang = np.arctan2(d[:, 1], d[:, 0])
    turn = np.abs((np.diff(ang) + np.pi) % (2 * np.pi) - np.pi) > np.radians(4)
    keep = np.r_[True, (sets[1:] != sets[:-1]) | turn, True]
    b = np.r_[sets, -1]
    return [[round(p[k, 0]), round(p[k, 1]), int(b[k])] for k in np.flatnonzero(keep)]


route_rows = []
for r, row in table.iterrows():
    route_rows.append({
        'id': r, 'name': row['name'], 'rank': int(row['rank']),
        'riders': round(row['riders']), 'ridersBench': round(row['riders_bench']) if pd.notna(row['riders_bench']) else None,
        'trips': int(row['trips']), 'vehHours': round(row['veh_hours'], 1), 'perHour': round(row['per_hour'], 1),
        'interval': round(row['interval'], 1) if pd.notna(row['interval']) else None, 'tier': row['tier'],
        'middayBph': round(row['midday_bph'], 1), 'lengthMi': round(row['length_mi'], 1),
        'path': export_path(r),
        'alt': [[[round(x), round(y)] for x, y in simplify(run, SIMPLIFY)] for run in alt.get(r, [])],
    })

ledger = Ledger(
    source=DATASET_NAME,
    dataset_id=DATASET_ID,
    url=DATASET_URL,
    pulled=pulled,
    units=f'average weekday riders per route, {WINDOW[0].date()} to {WINDOW[1].date()}; buses per route on {REF_DAY}',
    method=[
        'Weekday rides in the window / weekdays in the window; routes running part of the year are averaged over the whole year',
        f'Service from CTA GTFS (feed of {feed_date}) on Wednesday {REF_DAY}: trips, vehicle-hours, midday interval 10:00-15:00',
        'One shape per route, one direction; routes on a shared street drawn as offset parallels',
        'Coordinates in Chicago address units from State & Madison; the site plots them in miles',
    ],
    caveats=[
        'Riders are boardings on the whole route, not the load on any one street',
        f'Ridership runs to {WINDOW[1].date()}; service is the schedule of {REF_DAY}',
        'Quarter-mile coverage is straight-line distance, not a walk on streets',
    ],
)

payload = {
    'source': {'name': DATASET_NAME, 'id': DATASET_ID, 'url': DATASET_URL},
    'gtfs': {'url': GTFS_URL, 'feedDate': feed_date.isoformat(), 'refDay': REF_DAY.isoformat()},
    'window': [WINDOW[0].date().isoformat(), WINDOW[1].date().isoformat()],
    'benchmarkYear': BENCHMARK_YEAR,
    'summary': summary,
    'coverage': coverage,
    'tiers': [{'key': k, 'label': label, 'max': None if np.isinf(m) else m} for k, label, m in TIERS]
             + [{'key': NO_MIDDAY[0], 'label': NO_MIDDAY[1], 'max': None}],
    'notScheduled': [{'id': r, 'riders': round(v)} for r, v in not_scheduled.items()],
    'routes': route_rows,
    'bundles': [list(b) for b in bundles],
    'rail': rail,
}

path = write_plate(OUT_PATH, day=DAY, ledger=ledger, finding=FINDING, payload=payload)

# Read back and confirm the contract the website depends on.
written = json.loads(path.read_text())
assert {'asOf', 'finding', 'ledger', 'summary', 'routes', 'bundles'} <= written.keys()
assert len(written['routes']) == len(table)
assert all(-1 <= v[2] < len(bundles) for r in written['routes'] for v in r['path'])
log.info('Wrote %s (%.1f KB): %d routes, %d bundles, %d vertices', path.relative_to(DAY_DIR),
         path.stat().st_size / 1024, len(route_rows), len(bundles), sum(len(r['path']) for r in route_rows))

## Notes for the plate

- Finding: printed in section 5; it is generated from the data.
- Caveats for the ledger: riders are route boardings, not street loads; ridership and schedule dates
  differ; coverage is straight-line.
- Next: the website reads `output/plate.json`; the renderer draws each route on its streets by tier.